In [1]:
!pip install qiskit==1.4.3
!pip install qiskit_nature==0.7.2
!pip install qiskit_aer==0.17.1
!pip install pyscf
!pip install rdkit

In [2]:
!pip install qiskit_nature==0.7.2

In [3]:
from rdkit import Chem
from rdkit.Chem import AllChem

def cap_structure(input_pdb, output_pdb):
    """
    Reads a PDB file, identifies and caps dangling bonds with hydrogen atoms,
    and saves the capped structure to a new PDB file.
    """
    # Read the molecule from the PDB file, sanitizing it to fix valency issues
    mol = Chem.MolFromPDBFile(input_pdb, sanitize=True, removeHs=False)

    if mol is None:
        print(f"Error: Could not read molecule from {input_pdb}")
        return

    # Add hydrogens to all atoms to ensure full valency
    mol = Chem.AddHs(mol, explicitOnly=True)

    # Save the new molecule with the added hydrogens
    writer = Chem.PDBWriter(output_pdb)
    writer.write(mol)
    writer.close()

    print(f"Structure from '{input_pdb}' has been capped and saved to '{output_pdb}'")

if __name__ == "__main__":
    input_file = "res_17_21.pdb"
    output_file = "res_17_21_capped.pdb"
    cap_structure(input_file, output_file)

Structure from 'res_17_21.pdb' has been capped and saved to 'res_17_21_capped.pdb'


In [4]:
def convert_pdb_to_xyz(input_pdb, output_xyz):
    """
    Converts a PDB file to an XYZ file using RDKit.
    """
    # Read the molecule from the PDB file
    mol = Chem.MolFromPDBFile(input_pdb, sanitize=False, removeHs=False)

    if mol is None:
        print(f"Error: Could not read molecule from {input_pdb}")
        return

    # Create an XYZ file writer
    writer = Chem.PDBWriter(output_xyz)
    writer.write(mol)
    writer.close()

    # RDKit's PDBWriter can be tricky with the output format.
    # A more robust approach for XYZ is to build it manually:

    with open(output_xyz, 'w') as f:
        # Write the number of atoms
        f.write(f"{mol.GetNumAtoms()}\n\n")
        # Write each atom's symbol and coordinates
        for atom in mol.GetAtoms():
            pos = mol.GetConformer().GetAtomPosition(atom.GetIdx())
            symbol = atom.GetSymbol()
            f.write(f"{symbol}\t{pos.x:.4f}\t{pos.y:.4f}\t{pos.z:.4f}\n")

    print(f"Successfully converted '{input_pdb}' to '{output_xyz}'")

if __name__ == "__main__":
    # Replace these filenames with your actual file names
    input_file = "res_17_21_capped.pdb"
    output_file = "res_17_21_capped.xyz"
    convert_pdb_to_xyz(input_file, output_file)

Successfully converted 'res_17_21_capped.pdb' to 'res_17_21_capped.xyz'


In [5]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit_algorithms.minimum_eigensolvers import NumPyMinimumEigensolver, VQE
from qiskit_algorithms.optimizers import COBYLA, L_BFGS_B, SPSA, SLSQP
from qiskit_nature.second_q.transformers import FreezeCoreTransformer, ActiveSpaceTransformer
from qiskit_nature.second_q.formats.molecule_info import MoleculeInfo as Molecule
from qiskit_nature.second_q.mappers import ParityMapper
from qiskit_nature.units import DistanceUnit
from qiskit.circuit.library import TwoLocal
from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock
from qiskit_nature.second_q.drivers import PySCFDriver, MethodType
from qiskit_nature.second_q.algorithms import GroundStateEigensolver
from qiskit.circuit.library import EfficientSU2
from qiskit_aer import AerSimulator, Aer
from qiskit_aer.noise import NoiseModel
from qiskit_aer.primitives import Estimator
import qiskit_nature.settings
from pyscf import solvent, gto, scf
from pyscf.solvent import ddCOSMO

qiskit_nature.settings.use_pauli_sum_op = False


In [6]:
def create_abeta_driver(atom_data, charge, spin, basis='sto3g'):
  driver = PySCFDriver(
        atom=atom_data,
        unit=DistanceUnit.ANGSTROM,
        basis=basis,
        charge=charge,
        spin=spin,
        method=MethodType.RHF # or 'rohf' for odd-electron systems
    )
  return driver



def get_qubit_op(xyz_file_content, name):
  # Parse the XYZ content to get atom data
  lines = xyz_file_content.strip().split('\n')
  # Assuming the first line is charge and spin, and subsequent lines are atom data
  # We need to skip the first two lines (charge/spin and blank line) when passing to PySCF
  atom_data = lines[2:]


  if name == "wt":
    charge = -1
    spin = 2 # Changed spin to 2 for the WT fragment based on common practice for radicals
    driver = create_abeta_driver(atom_data, charge, spin)
  else:
    charge = 0 # Assuming neutral for the mutant fragment
    spin = 1 # Assuming singlet for the mutant fragment
    driver = create_abeta_driver(atom_data, charge, spin)

  # Create PySCF molecule object for COSMO calculation
  if name == "wt":
    mol = gto.M(atom='\n'.join(atom_data), basis='sto-3g', charge=-1, spin=2) # Changed spin to 2 here as well
  else:
    mol = gto.M(atom='\n'.join(atom_data), basis='sto-3g', charge=0, spin=1)

  mf = scf.RHF(mol)
  cosmo = solvent.ddCOSMO(mol)
  mf = solvent._attach_solvent._for_scf(mf, cosmo)
  mf.with_solvent.eps = 80.0

  problem = driver.run()

  # Define active space based on fragment type
  if name == "wt":
    transformer = ActiveSpaceTransformer(
     num_electrons=6,   # 4 π-electrons (COO⁻) + 2 lone pairs
     num_spatial_orbitals=5,
     active_orbitals=[8, 9, 10, 11, 12]  # Indices covering:
  )
  else:
    transformer = ActiveSpaceTransformer(
      num_electrons=4,   # Backbone lone pairs (N-H, C=O)
      num_spatial_orbitals=4,
      active_orbitals=[6, 7, 8, 9]  # Indices for:
    )
  problem = transformer.transform(problem)
  num_particles = problem.num_particles
  num_spatial_orbitals = problem.num_spatial_orbitals
  mapper = ParityMapper(num_particles=num_particles)
  hamiltonian = mapper.map(problem.second_q_ops()[0])
  return hamiltonian, num_particles, num_spatial_orbitals, problem, mapper

In [7]:
def exact_solver(hamiltonian, problem):
  sol = NumPyMinimumEigensolver().compute_minimum_eigenvalue(hamiltonian)
  result = problem.interpret(sol)
  return result

In [8]:
distances = np.arange(0.9, 1.2,0.02)
exact_energies = []
vqe_en = []
optimizer = SLSQP(maxiter = 10)
noise = Estimator(approximation = True)

In [9]:
"""import pyscf
for dist in distances:
  (hamiltonian, num_particles, num_spatial_orbitals, problem, mapper) = get_qubit_op()
  result = exact_solver(hamiltonian, problem)
  exact_energies.append(result.total_energies[0].real)

  init_state = HartreeFock(
      num_spatial_orbitals = num_spatial_orbitals,
      num_particles = num_particles,
      qubit_mapper = mapper
  )

  ansatz = UCCSD(
      num_spatial_orbitals=num_spatial_orbitals,
      num_particles=num_particles,
      qubit_mapper=mapper,
      initial_state=init_state,
  )

  vqe = VQE(
      estimator = noise,
      optimizer = optimizer,
      ansatz = ansatz,
      initial_point = [0] * int(ansatz.num_parameters),
  )

  vqe_calc = vqe.compute_minimum_eigenvalue(hamiltonian)
  vqe_result = problem.interpret(vqe_calc).total_energies[0].real
  vqe_en.append(vqe_result)

print(f"dist: {dist} - Exact En:{result.total_energies[0].real}")"""

'import pyscf\nfor dist in distances:\n  (hamiltonian, num_particles, num_spatial_orbitals, problem, mapper) = get_qubit_op()\n  result = exact_solver(hamiltonian, problem)\n  exact_energies.append(result.total_energies[0].real)\n\n  init_state = HartreeFock(\n      num_spatial_orbitals = num_spatial_orbitals,\n      num_particles = num_particles,\n      qubit_mapper = mapper\n  )\n\n  ansatz = UCCSD(\n      num_spatial_orbitals=num_spatial_orbitals,\n      num_particles=num_particles,\n      qubit_mapper=mapper,\n      initial_state=init_state,\n  )\n\n  vqe = VQE(\n      estimator = noise,\n      optimizer = optimizer,\n      ansatz = ansatz,\n      initial_point = [0] * int(ansatz.num_parameters),\n  )\n\n  vqe_calc = vqe.compute_minimum_eigenvalue(hamiltonian)\n  vqe_result = problem.interpret(vqe_calc).total_energies[0].real\n  vqe_en.append(vqe_result)\n\nprint(f"dist: {dist} - Exact En:{result.total_energies[0].real}")'

In [10]:
def read_xyz_file(filepath, charge, mult):
    with open(filepath, 'r') as f:
        lines = f.readlines()[2:]  # Skip atom count and comment
    mol_lines = [f"{charge} {mult}"]
    for line in lines:
        parts = line.split()
        if len(parts) < 4:
            continue
        atom = parts[0]
        x, y, z = map(float, parts[1:4])
        mol_lines.append(f"{atom} {x:.6f} {y:.6f} {z:.6f}")
    return "\n".join(mol_lines)

wt_xyz_content = read_xyz_file("res_17_21_capped.xyz", -1, 2)
#arctic_xyz_content = read_xyz_file("", 0, 1)
print(wt_xyz_content)
#print(f"Arctic: {arctic_xyz_content}")

-1 2
N 94.950000 40.230000 30.400000
H 94.230000 39.530000 30.470000
C 95.510000 40.500000 29.060000
H 96.570000 40.300000 28.900000
C 94.920000 39.450000 28.110000
H 93.830000 39.460000 28.110000
H 95.180000 38.480000 28.530000
C 95.440000 39.420000 26.670000
H 95.580000 40.360000 26.140000
C 96.820000 38.820000 26.490000
H 96.760000 37.750000 26.670000
H 97.160000 39.020000 25.480000
H 97.460000 39.330000 27.210000
C 94.600000 38.590000 25.740000
H 94.970000 38.480000 24.720000
H 94.490000 37.610000 26.220000
H 93.620000 39.040000 25.570000
C 95.250000 41.940000 28.610000
O 94.180000 42.260000 28.100000
N 96.270000 42.820000 28.670000
H 97.210000 42.480000 28.810000
C 96.200000 44.200000 28.140000
H 95.360000 44.250000 27.440000
C 96.080000 45.240000 29.220000
H 96.240000 46.260000 28.870000
C 94.660000 45.310000 29.650000
H 94.270000 44.380000 30.050000
H 94.520000 46.000000 30.480000
H 94.070000 45.720000 28.830000
C 96.950000 45.060000 30.510000
H 96.730000 45.800000 31.280000
H 9

In [11]:
# Load XYZ files (replace paths with actual file paths)
#arctic_xyz_content="C -1.315400 1.885570 0.156880;N -0.202920 -1.302470 1.306590;H -0.961460 -0.677200 1.034640;C 0.976350 -0.519810 1.679580;H 0.902440 0.506890 1.312870;H 1.875680 -0.962010 1.258000;C 1.123680 -0.495220 3.179730;O 2.167710 -0.779570 3.763390;C 2.871270 -2.342300 -0.899730;H -0.627080 2.198190 0.945260;H -0.828050 1.154350 -0.488600;H -1.603480 2.754770 -0.435660;H -2.206760 1.439060 0.609250;H 0.010520 -1.847540 0.470540;H 0.245470 -0.130140 3.742670;H 3.750850 -2.061660 -1.479180;H 2.078390 -1.606800 -1.068710;H 3.134600 -2.371500 0.159900;H 2.521180 -3.327860 -1.208390;"
#wt_xyz_content="C 0.655810 2.228690 -0.260920;N -0.052710 -1.897580 1.254060;H 0.938280 -1.724810 1.434880;C -0.239590 -2.163740 -0.174500;H 0.114870 -1.318430 -0.770650;H -1.298500 -2.327070 -0.391130;C 0.520920 -3.414660 -0.561210;O 1.368510 -3.444960 -1.453300;C 2.889860 -0.482610 -0.583600;H 1.706560 2.213110 -0.558090;H 0.166400 1.312640 -0.598970;H 0.166290 3.092230 -0.714710;H 0.587810 2.295910 0.827000;H -0.524740 -1.023700 1.489260;H 0.246490 -4.332710 -0.009800;H 3.798310 0.112330 -0.692600;H 2.298490 -0.413450 -1.498920;H 2.307960 -0.106400 0.259430;H 3.154900 -1.525980 -0.398330;"
wt_hamiltonian, wt_particles, wt_orbitals, wt_problem, wt_mapper = get_qubit_op(wt_xyz_content, name="wt")
#arctic_hamiltonian, arctic_particles, arctic_orbitals, arctic_problem, arctic_mapper = get_qubit_op(arctic_xyz_content, name="mut")

# Run VQE for both
def run_vqe(hamiltonian, problem, mapper, num_particles, num_orbitals):
    #ansatz = TwoLocal(
    #    num_spatial_orbitals=problem.num_spatial_orbitals,
    #    num_particles=problem.num_particles,
    #    qubit_mapper=mapper,
    #    initial_state=HartreeFock(problem.num_spatial_orbitals, problem.num_particles, mapper)
    #)
    num_qubits = hamiltonian.num_qubits
    init_state = HartreeFock(num_qubits, problem.num_particles, mapper)
    ansatz = TwoLocal(
        num_qubits=num_qubits,
        rotation_blocks=['ry', 'rz'],
        entanglement_blocks='cx',
        entanglement='linear',
        reps=2,
        insert_barriers=True,
    )
    vqe = VQE(
        estimator=Estimator(),
        ansatz=ansatz,
        optimizer=COBYLA(maxiter=100)
    )
    result = vqe.compute_minimum_eigenvalue(hamiltonian)
    return problem.interpret(result)

wt_result = run_vqe(wt_hamiltonian, wt_problem, wt_mapper, wt_particles, wt_orbitals)
#arctic_result = run_vqe(arctic_hamiltonian, arctic_problem, arctic_mapper, arctic_particles, arctic_orbitals)

# Compare energies
delta_energy =  wt_result.total_energies[0]
print(f"ΔE (Arctic-WT) = {delta_energy} hartree")  # Convert to kcal/mol

/tmp/ipykernel_71463/4090211400.py:33: DeprecationWarning: Estimator has been deprecated as of Aer 0.15, please use EstimatorV2 instead.
  wt_result = run_vqe(wt_hamiltonian, wt_problem, wt_mapper, wt_particles, wt_orbitals)
/tmp/ipykernel_71463/4090211400.py:33: DeprecationWarning: Option approximation=False is deprecated as of qiskit-aer 0.13. It will be removed no earlier than 3 months after the release date. Instead, use BackendEstimator from qiskit.primitives.
  wt_result = run_vqe(wt_hamiltonian, wt_problem, wt_mapper, wt_particles, wt_orbitals)


ΔE (Arctic-WT) = -1766.7234315736305 hartree
